# Serverless Private Git — Connectivity Debug Notebook

This notebook validates network connectivity from **serverless compute** to your Git remote.
It must be run on a **serverless** compute resource (not a classic cluster).

**What this notebook does:**
1. Detects whether the Git CLI preview is enabled in your workspace.
2. Reads your `/Workspace/.git_settings/config.json` (if present) and resolves settings for your remote.
3. Tests DNS resolution for the Git server hostname.
4. Tests HTTPS connectivity to the Git server.
   - If Git CLI preview is enabled, uses `git ls-remote`.
   - Otherwise, uses `curl` with settings from `config.json`.

**To use:** Edit `REMOTE_URL` in the cell below, then **Run All**.

In [0]:
# Edit this URL to point to your Git remote:
REMOTE_URL = "https://your-git-server.company.com/org/repo.git"

## Setup

In [0]:
import json
import os
import subprocess
import socket
import tempfile
from urllib.parse import urlparse

remote_url = REMOTE_URL.strip()
if not remote_url or remote_url == "https://your-git-server.company.com/org/repo.git":
    dbutils.notebook.exit("ERROR: Please edit REMOTE_URL in the cell above to your Git remote URL and re-run.")

parsed = urlparse(remote_url)
if parsed.scheme != "https":
    dbutils.notebook.exit(f"ERROR: URL scheme must be https (got '{parsed.scheme}'). Provide a full URL like https://git.company.com/org/repo.git")

git_host = parsed.hostname
print(f"Target remote : {remote_url}")
print(f"Hostname      : {git_host}")

In [ ]:
is_serverless = os.getenv("IS_SERVERLESS", "").lower() == "true"

if not is_serverless:
    dbutils.notebook.exit("ERROR: This notebook must be run on serverless compute. Please re-attach to a serverless compute resource and re-run.")

print("✅ Running on serverless compute.")

## Step 1 — Detect Git CLI Availability

The Git CLI preview changes how this notebook tests connectivity. When enabled, the notebook uses `git ls-remote` (which exercises the full Git authentication and transport stack). When disabled, it falls back to `curl` against the Git smart-HTTP endpoint. This step determines which path to use.

In [0]:
git_cli_available = False
git_config_global = os.environ.get("GIT_CONFIG_GLOBAL", "")

if git_config_global:
    try:
        with open(git_config_global, "r") as f:
            git_config_content = f.read().strip()
        if git_config_content:
            git_cli_available = True
            print(f"✅ Git CLI preview is enabled.")
            print(f"   GIT_CONFIG_GLOBAL = {git_config_global}")
            print("   Connectivity test will use: git ls-remote")
        else:
            print("ℹ️  Git CLI preview is not enabled (virtual git config is empty).")
            print("   Connectivity test will use: curl")
    except FileNotFoundError:
        print(f"ℹ️  Git CLI preview is not enabled ({git_config_global} not found).")
        print("   Connectivity test will use: curl")
    except Exception as e:
        print(f"ℹ️  Could not read virtual git config: {e}")
        print("   Connectivity test will use: curl")
else:
    print("ℹ️  GIT_CONFIG_GLOBAL is not set. Git CLI preview is not enabled.")
    print("   Connectivity test will use: curl")

## Step 2 — Load config.json

In [0]:
CONFIG_PATH = "/Workspace/.git_settings/config.json"

config = None
config_error = None
try:
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)
    print(f"✅ Loaded config from {CONFIG_PATH}\n")
    print(json.dumps(config, indent=2))
except FileNotFoundError:
    print(f"ℹ️  No config file found at {CONFIG_PATH}")
    print("   Default settings will be used (SSL verification enabled, no proxy).")
except json.JSONDecodeError as e:
    config_error = str(e)
    print(f"❌ config.json exists but contains invalid JSON: {e}")
except Exception as e:
    config_error = str(e)
    print(f"❌ Error reading config.json: {e}")

## Step 3 — Resolve Configuration for Target Remote

This step determines the effective SSL, proxy, CA certificate, and custom port settings for your remote URL by merging defaults with any matching remote-specific overrides from `config.json`. Incorrect settings here are a common source of connectivity failures — verify that the resolved values match what your Git server expects.

In [0]:
resolved_ssl_verify = True
resolved_ca_cert = ""
resolved_proxy = ""
resolved_port = ""

if config:
    defaults = config.get("default", {})
    resolved_ssl_verify = defaults.get("sslVerify", True)
    resolved_ca_cert = defaults.get("caCertPath", "")
    resolved_proxy = defaults.get("httpProxy", "")
    resolved_port = str(defaults.get("customHttpPort", ""))

    matched_remote = None
    for remote_entry in config.get("remotes", []):
        prefix = remote_entry.get("urlPrefix", "")
        if prefix and remote_url.startswith(prefix):
            if matched_remote is None or len(prefix) > len(matched_remote.get("urlPrefix", "")):
                matched_remote = remote_entry

    if matched_remote:
        print(f"✅ Matched remote config with urlPrefix: {matched_remote['urlPrefix']}\n")
        if "sslVerify" in matched_remote:
            resolved_ssl_verify = matched_remote["sslVerify"]
        if "caCertPath" in matched_remote:
            resolved_ca_cert = matched_remote["caCertPath"]
        if "httpProxy" in matched_remote:
            resolved_proxy = matched_remote["httpProxy"]
        if "customHttpPort" in matched_remote:
            resolved_port = str(matched_remote["customHttpPort"])
    else:
        print("ℹ️  No remote-specific config matched. Using defaults.\n")

print(f"Resolved settings for {remote_url}:")
print(f"  sslVerify      : {resolved_ssl_verify}")
print(f"  caCertPath     : {resolved_ca_cert or '(none)'}")
print(f"  httpProxy      : {resolved_proxy or '(none)'}")
print(f"  customHttpPort : {resolved_port or '(default)'}")

if resolved_ca_cert:
    if os.path.exists(resolved_ca_cert):
        print(f"\n✅ CA certificate file exists at {resolved_ca_cert}")
    else:
        print(f"\n❌ CA certificate file NOT found at {resolved_ca_cert}")
        print("   Ensure the file exists and that Git users have View permissions.")

## Step 4 — DNS Resolution

This tests whether the Git server hostname can be resolved from within serverless compute. DNS failures typically mean the NCC/PrivateLink configuration is missing or hasn't propagated yet (allow 10+ minutes after changes).

In [0]:
dns_port = int(resolved_port) if resolved_port else 443
print(f"Resolving hostname: {git_host} (port {dns_port})\n")
try:
    addr_info = socket.getaddrinfo(git_host, dns_port)
    resolved_ips = sorted(set(info[4][0] for info in addr_info))
    print(f"✅ DNS resolution succeeded.")
    for ip in resolved_ips:
        print(f"   {git_host} → {ip}")
except socket.gaierror as e:
    print(f"❌ DNS resolution failed for {git_host}: {e}")
    print("   Possible causes:")
    print("   - The hostname is incorrect.")
    print("   - DNS is not configured to resolve this private hostname from serverless compute.")
    print("   - The NCC / PrivateLink endpoint may not be set up correctly.")

## Step 5 — Connectivity Test

In [0]:
def _build_test_url():
    url = remote_url.rstrip("/")
    if resolved_port:
        p = urlparse(url)
        url = f"{p.scheme}://{p.hostname}:{resolved_port}{p.path}"
    return url + "/info/refs?service=git-upload-pack"

def _build_curl_flags():
    flags = []
    if not resolved_ssl_verify:
        flags.append("--insecure")
    if resolved_ca_cert:
        flags.extend(["--cacert", resolved_ca_cert])
    if resolved_proxy:
        flags.extend(["--proxy", resolved_proxy])
    return flags

if git_cli_available:
    print("Using Git CLI to test connectivity...\n")
    print(f"  Command: git ls-remote --heads {remote_url}\n")

    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"

    try:
        result = subprocess.run(
            ["git", "ls-remote", "--heads", remote_url],
            capture_output=True, text=True, timeout=60, env=env,
        )
        if result.returncode == 0:
            head_count = len(result.stdout.strip().splitlines()) if result.stdout.strip() else 0
            print(f"✅ git ls-remote succeeded — {head_count} branch(es) found.")
            if result.stdout.strip():
                for line in result.stdout.strip().splitlines()[:10]:
                    print(f"   {line}")
                if head_count > 10:
                    print(f"   ... and {head_count - 10} more")
        else:
            print(f"❌ git ls-remote failed (exit code {result.returncode})")
            if result.stderr.strip():
                print(f"\n   stderr:\n")
                for line in result.stderr.strip().splitlines():
                    print(f"   {line}")
            print("\n   The server is reachable but denied access. Check the stderr above for details.")
            print("   Common causes: missing/invalid Git credentials, IP allowlist restrictions,")
            print("   or repository-level access policies.")
    except subprocess.TimeoutExpired:
        print("❌ git ls-remote timed out after 60 seconds.")
        print("   The server may be unreachable from serverless compute.")
        print("   Check your NCC and PrivateLink configuration.")
    except Exception as e:
        print(f"❌ Error running git ls-remote: {e}")

else:
    test_url = _build_test_url()

    body_file = tempfile.NamedTemporaryFile(mode="w+", suffix=".txt", delete=False)
    body_file.close()

    curl_cmd = [
        "curl", "-sS",
        "-o", body_file.name,
        "-w", "%{http_code} %{time_connect} %{time_total}",
        "--max-time", "30",
    ] + _build_curl_flags() + [test_url]

    print("Using curl to test connectivity...\n")
    print(f"  Command: {' '.join(curl_cmd)}\n")

    try:
        result = subprocess.run(
            curl_cmd,
            capture_output=True, text=True, timeout=35,
        )
        if result.returncode == 0 and result.stdout.strip():
            parts = result.stdout.strip().split()
            http_code = parts[0] if len(parts) > 0 else "?"
            time_connect = parts[1] if len(parts) > 1 else "?"
            time_total = parts[2] if len(parts) > 2 else "?"

            code_int = int(http_code) if http_code.isdigit() else 0
            if 200 <= code_int < 400:
                print(f"✅ Connection succeeded.")
            elif code_int in (401, 403):
                print(f"⚠️  Server is reachable but returned HTTP {http_code}.")
                print(f"   Network connectivity is working, but the server denied access.")
                print(f"   Check the response body below for details. Common causes: missing/invalid")
                print(f"   Git credentials, IP allowlist restrictions, or repository-level access policies.")
            elif code_int == 404:
                print(f"⚠️  Server is reachable but returned HTTP 404.")
                print(f"   The repository URL may be incorrect, or the server does not support smart HTTP.")
            else:
                print(f"❌ Unexpected HTTP response code: {http_code}")

            print(f"\n   HTTP status   : {http_code}")
            print(f"   Connect time  : {time_connect}s")
            print(f"   Total time    : {time_total}s")

            if code_int >= 400:
                try:
                    with open(body_file.name, "r") as f:
                        body = f.read().strip()
                    if body:
                        print(f"\n   Response body:\n")
                        for line in body.splitlines()[:20]:
                            print(f"   {line}")
                        if len(body.splitlines()) > 20:
                            print(f"   ... ({len(body.splitlines()) - 20} more lines)")
                except Exception:
                    pass
        else:
            print(f"❌ curl failed (exit code {result.returncode})")
            if result.stderr.strip():
                print(f"\n   stderr:\n")
                for line in result.stderr.strip().splitlines():
                    print(f"   {line}")

            if "Could not resolve host" in (result.stderr or ""):
                print("\n   DNS resolution failed. Check hostname and NCC configuration.")
            elif "Connection refused" in (result.stderr or ""):
                print("\n   Connection refused. The server may not be listening on the expected port.")
            elif "Connection timed out" in (result.stderr or "") or "timed out" in (result.stderr or "").lower():
                print("\n   Connection timed out. Check PrivateLink and NCC configuration.")
            elif "SSL" in (result.stderr or "") or "certificate" in (result.stderr or "").lower():
                print("\n   SSL/TLS error. Check sslVerify and caCertPath settings in config.json.")

    except subprocess.TimeoutExpired:
        print("❌ curl timed out after 30 seconds.")
        print("   The server may be unreachable from serverless compute.")
        print("   Check your NCC and PrivateLink configuration.")
    except Exception as e:
        print(f"❌ Error running curl: {e}")
    finally:
        try:
            os.unlink(body_file.name)
        except Exception:
            pass

## Step 6 — Verbose Connectivity Details (for Databricks Support)

This section collects detailed diagnostic output for troubleshooting. If you need to file a support ticket, copy the output below or export this notebook as a DBC archive and share it with Databricks Support.

In [ ]:
print("=" * 60)
print("CONNECTIVITY DEBUG SUMMARY")
print("=" * 60)
print(f"Remote URL         : {remote_url}")
print(f"Hostname           : {git_host}")
print(f"Serverless compute : {is_serverless}")
print(f"Git CLI available  : {git_cli_available}")
print(f"config.json loaded : {config is not None}")
if config_error:
    print(f"config.json error  : {config_error}")
print(f"SSL verify         : {resolved_ssl_verify}")
print(f"CA cert path       : {resolved_ca_cert or '(none)'}")
print(f"HTTP proxy         : {resolved_proxy or '(none)'}")
print(f"Custom HTTP port   : {resolved_port or '(default)'}")
print("=" * 60)

if not git_cli_available:
    print("\nVerbose curl output for support:\n")
    verbose_cmd = ["curl", "-vvv", "--max-time", "15"] + _build_curl_flags() + [_build_test_url()]

    try:
        result = subprocess.run(verbose_cmd, capture_output=True, text=True, timeout=20)
        for line in (result.stderr or "").splitlines():
            print(f"  {line}")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("\nVerbose git output for support:\n")
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    # GIT_TRACE_CURL redacts sensitive headers (Authorization, Cookie, and the
    # HTTP/2 h2/h2h3 variants) by default, so this output is safe to share with
    # support. Avoid GIT_CURL_VERBOSE, which is less consistent about redaction.
    env["GIT_TRACE_CURL"] = "1"
    env["GIT_TRACE"] = "1"

    try:
        result = subprocess.run(
            ["git", "ls-remote", "--heads", remote_url],
            capture_output=True, text=True, timeout=30, env=env,
        )
        for line in (result.stderr or "").splitlines():
            print(f"  {line}")
    except Exception as e:
        print(f"  Error: {e}")

## Troubleshooting Guide

| Symptom | Likely Cause | Action |
|---------|-------------|--------|
| DNS resolution failed | Hostname not resolvable from serverless | Verify the NCC private endpoint rule includes the correct FQDN. Wait 10+ minutes after NCC changes. |
| Connection timed out | No network path to server | Verify PrivateLink / NCC is configured and endpoint connection is approved. |
| Connection refused | Server not listening on expected port | Check `customHttpPort` in config.json. Verify the Git server is running. |
| SSL/TLS certificate error | Self-signed or private CA cert | Set `caCertPath` in `/Workspace/.git_settings/config.json` to your CA cert, or set `sslVerify` to `false` for testing. |
| HTTP 401/403 | Connectivity works, server denied access | Check the response body and stderr for details. Common causes: missing/invalid Git credentials, IP allowlist restrictions, or repository-level access policies. |
| HTTP 404 | Wrong URL or server doesn't support smart HTTP | Verify the repository URL is correct and includes the full path. |
| config.json not found | No custom config | This is fine if your Git server uses standard HTTPS on port 443 with a public CA. |

If the issue persists, export this notebook as a DBC archive and share with Databricks Support.